# ModernBERT Binary SFT: LLM Routing

This notebook demonstrates the **Supervised Fine-Tuning (SFT)** of a **ModernBERT** model for **Binary Classification** of user prompts. Specifically, it uses the **LLM Router Dataset** to determine whether a query is best suited for a 'Large LLM' vs a 'Smaller/Local Model'.

The goal is to adapt **ModernBERT**—a state-of-the-art encoder model—to a specific downstream task where the output is a single binary label. This process involves:
- Loading and preprocessing the **LLM Router synthetic dataset**.
- Tokenizing text sequences specifically for ModernBERT's architecture.
- Fine-tuning the model for binary sequence classification.
- Evaluating model performance using metrics such as **Accuracy and F1-score**.


## 1. Setup environment

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

!uv add loguru matplotlib torchvista datasets accelerate hf-transfer

Resolved 186 packages in 19ms
Audited 173 packages in 0.20ms


In [ ]:
import os
import time
from loguru import logger
from random import randrange
import matplotlib.pyplot as plt

print = logger.info

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
import torch
from torchvista import trace_model
from datasets import load_dataset, logging
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainerCallback,
    Trainer,
    TrainingArguments,
    pipeline,
)

logging.set_verbosity_info()

## 2. Load dataset and Preparetion

In [ ]:
dataset_id = "DevQuasar/llm_router_dataset-synth"

raw_dataset = load_dataset(dataset_id)
print(f"Train dataset size: {len(raw_dataset['train'])}")
print(f"Test dataste size: {len(raw_dataset['test'])}")

In [ ]:
random_id = randrange(len(raw_dataset["train"]))
raw_dataset["train"][random_id]

In [ ]:
model_id = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.model_max_length = 512

In [ ]:
column_names = raw_dataset["train"].column_names

# 1. Rename label column
renamed_dataset = (
    raw_dataset.rename_column("label", "labels")
    if "label" in column_names
    else raw_dataset
)


# 2. Tokenize
def tokenize(batch):
    return tokenizer(
        batch["prompt"], padding="max_length", truncation=True, return_tensors="pt"
    )


tokenized_dataset = renamed_dataset.map(
    tokenize,
    batched=True,
    remove_columns=[col for col in ["prompt"] if col in column_names],
)
print(tokenized_dataset["train"].features.keys())

## 3. Finetuning

In [ ]:
labels = tokenized_dataset["train"].features["labels"].names

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    attn_implementation="eager",
    reference_compile=False,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
)

In [ ]:
dummy_input = torch.zeros((1, 512), dtype=torch.long).to(
    next(model.parameters()).device
)
graph = trace_model(model, dummy_input)
graph

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    score = f1_score(labels, predictions, labels=[0, 1], average="weighted")
    return {"f1": score}

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./moderbert-llm-router/runs

In [ ]:
class SpeedProfilingCallback(TrainerCallback):

    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"🚀 Training started | Device: {args.device} | BF16: {args.bf16}")

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0 and state.global_step > 0:
            elapsed = time.time() - self.start_time
            it_per_sec = state.global_step / elapsed
            print(
                f"⏱️ [Step {state.global_step}] Throughput: {it_per_sec:.2f} it/s | Elapsed: {elapsed:.1f}s",
                flush=True,
            )

    def on_train_end(self, args, state, control, **kwargs):
        final_time = time.time() - self.start_time
        avg_speed = state.global_step / final_time
        print(
            f"\n✨ [FINAL SPEED REPORT] Total Duration: {final_time:.2f}s | Final Avg Speed: {avg_speed:.2f} it/s"
        )

In [ ]:
class LossPrinterCallback(TrainerCallback):

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(
                f"📊 [Step {state.global_step}] | Loss: {logs['loss']:.4f} | LR: {logs.get('learning_rate', 'N/A')}",
                flush=True,
            )

In [ ]:
class SurveillanceCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0 and state.global_step > 0:
            model = kwargs.get("model")
            for name, param in model.named_parameters():
                if "classifier" in name and "weight" in name:
                    norm = param.norm().item()
                    grad_norm = (
                        param.grad.norm().item() if (param.grad is not None) else 0.0
                    )
                    print(
                        f"📈 [Step {state.global_step}] Classifier Norm: {norm:.6f} | Grad Norm: {grad_norm:.6f}",
                        flush=True,
                    )
                    if grad_norm == 0:
                        print("⚠️ WARNING: Zero Gradient detected!", flush=True)
                    if torch.isnan(param).any():
                        print("❌ CRITICAL: NaN detected in weights!", flush=True)

In [ ]:
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 2
EPOCHS = 5

total_steps = (
    len(tokenized_dataset["train"]) // (BATCH_SIZE * GRAD_ACCUM_STEPS)
) * EPOCHS
warmup_steps = int(total_steps * 0.1)

training_args = TrainingArguments(
    output_dir="moderbert-llm-router",
    learning_rate=5e-6,
    warmup_steps=warmup_steps,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    max_grad_norm=0.3,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    optim="adamw_torch_fused",
    bf16=True,
    fp16=False,
    metric_for_best_model="eval_loss",
    eval_strategy="steps",
    eval_steps=100,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="tensorboard",
    report_to="tensorboard",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
    callbacks=[SpeedProfilingCallback(), LossPrinterCallback(), SurveillanceCallback()],
)

In [ ]:
trainer.train()

In [23]:
tokenizer.save_pretrained("moderbert-llm-router")
model.save_pretrained("moderbert-llm-router")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 4. Inference

In [ ]:
classifier = pipeline(
    "text-classification",
    model="moderbert-llm-router",
    device=0,
)

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

2026-02-03 09:45:38.894 | INFO     | __main__:<module>:10 - [{'label': 'large_llm', 'score': 0.9999922513961792}]


In [32]:
sample = "How does the structure and function of plasmodesmata affect cell-to-cell communication and signaling in plant tissues, particularly in response to environmental stresses?"

pred = classifier(sample)
print(pred)

2026-02-03 09:45:48.328 | INFO     | __main__:<module>:4 - [{'label': 'large_llm', 'score': 0.9999922513961792}]
